In [35]:
import pandas as pd

def load_data(file_path):
    try:
        df = pd.read_excel(file_path)
        df['year'] = df['year'].astype(int)
        df['produced_material'] = df['produced_material'].astype(str)
        df['component_material'] = df['component_material'].astype(str).replace('nan', pd.NA)
        df['plant_id'] = df['plant_id'].astype(str)
        print("Rows:", len(df))
        print(df.head())
        return df
    except FileNotFoundError:
        print(f"Error: File {file_path} not found")
        return None
    except Exception as e:
        print(f"Error loading file: {e}")
        return None

def aggregate_data(df):
    try:
        df_agg = df.groupby([
            'plant_id', 'year', 'produced_material', 'component_material',
            'produced_material_release_type', 'produced_material_production_type',
            'component_material_release_type', 'component_material_production_type'
        ], dropna=False).agg({
            'produced_material_quantity': 'sum',
            'component_material_quantity': 'sum'
        }).reset_index()
        print("Rows aggregated:", len(df_agg))
        return df_agg
    except Exception as e:
        print(f"Error in aggregation: {e}")
        return None

def identify_fin_materials(df_agg):
    try:
        fin_materials = set(df_agg[df_agg['produced_material_release_type'] == 'FIN']['produced_material'])
        print("FIN materials:", fin_materials)
        return fin_materials
    except Exception as e:
        print(f"Error identifying FIN materials: {e}")
        return set()

def create_lookup_dictionaries(df_agg):
    try:
        prod_quantities = df_agg.groupby(['plant_id', 'year', 'produced_material'])['produced_material_quantity'].sum().to_dict()
        component_df = df_agg[df_agg['component_material'].notna()]
        
        lookup_dicts = {
            'release_type': df_agg[['produced_material', 'produced_material_release_type']]
                .drop_duplicates('produced_material')
                .set_index('produced_material')['produced_material_release_type'].to_dict(),
            'prod_type': df_agg[['produced_material', 'produced_material_production_type']]
                .drop_duplicates('produced_material')
                .set_index('produced_material')['produced_material_production_type'].to_dict(),
            'comp_release_type': component_df[['component_material', 'component_material_release_type']]
                .drop_duplicates('component_material')
                .set_index('component_material')['component_material_release_type'].to_dict(),
            'comp_prod_type': component_df[['component_material', 'component_material_production_type']]
                .drop_duplicates('component_material')
                .set_index('component_material')['component_material_production_type'].to_dict(),
            'components': component_df.groupby('produced_material', group_keys=False).apply(
                lambda x: dict(zip(x['component_material'], x['component_material_quantity'])),
                include_groups=False
            ).to_dict(),
            'prod_quantity': prod_quantities
        }
        return lookup_dicts
    except Exception as e:
        print(f"Error creating lookup dictionaries: {e}")
        return {}

def build_hierarchy(fin_material, plant, year, lookup_dicts, result_list, current_material=None):
    if current_material is None:
        components = lookup_dicts['components'].get(fin_material, {})
        for component, comp_quantity in components.items():
            fin_release_type = lookup_dicts['release_type'].get(fin_material, 'Unknown')
            comp_release_type = lookup_dicts['comp_release_type'].get(component, 'Unknown')
            build_hierarchy(fin_material, plant, year, lookup_dicts, result_list, component)
    else:
        fin_release_type = lookup_dicts['release_type'].get(fin_material, 'Unknown')
        current_release_type = lookup_dicts['release_type'].get(current_material, 'Unknown')
        if fin_release_type != current_release_type:
            components = lookup_dicts['components'].get(current_material, {})
            for component, comp_quantity in components.items():
                comp_release_type = lookup_dicts['comp_release_type'].get(component, 'Unknown')
                row = {
                    'plant': plant,
                    'year': year,
                    'fin_material_id': fin_material,
                    'fin_material_release_type': fin_release_type,
                    'fin_material_production_type': lookup_dicts['prod_type'].get(fin_material, None),
                    'fin_production_quantity': lookup_dicts['prod_quantity'].get((plant, year, fin_material), 0),
                    'prod_material_id': current_material,
                    'prod_material_release_type': current_release_type,
                    'prod_material_production_type': lookup_dicts['prod_type'].get(current_material, None),
                    'prod_material_production_quantity': lookup_dicts['prod_quantity'].get((plant, year, current_material), 0),
                    'component_id': component,
                    'component_material_release_type': comp_release_type,
                    'component_material_production_type': lookup_dicts['comp_prod_type'].get(component, None),
                    'component_consumption_quantity': comp_quantity
                }
                result_list.append(row)
                build_hierarchy(fin_material, plant, year, lookup_dicts, result_list, component)

def main():
    file_path = 'C:/Users/egor2/trainee_de/task2_data/task_2_data_ex.xlsx'

    df = load_data(file_path)
    if df is None:
        return

    df_agg = aggregate_data(df)
    if df_agg is None:
        return

    fin_materials = identify_fin_materials(df_agg)
    if not fin_materials:
        print("No FIN materials found")
        return

    lookup_dicts = create_lookup_dictionaries(df_agg)
    if not lookup_dicts:
        print("Failed to create lookup dictionaries")
        return

    fin_material_combinations = df_agg[df_agg['produced_material'].isin(fin_materials)][
        ['plant_id', 'year', 'produced_material']
    ].drop_duplicates()
    print("FIN material combinations:", len(fin_material_combinations))
    print(fin_material_combinations)

    result_list = []
    for _, row in fin_material_combinations.iterrows():
        plant = row['plant_id']
        year = row['year']
        fin_material = row['produced_material']
        build_hierarchy(fin_material, plant, year, lookup_dicts, result_list)

    result_df = pd.DataFrame(result_list)
    print("Total rows:", len(result_df))
    print(result_df.head())
    print("Rows for year 2000:", len(result_df[result_df['year'] == 2000]))
    print("Rows for plant PLANT_15:", len(result_df[result_df['plant'] == 'PLANT_15']))

    result_df.to_excel('bom_explosion_output.xlsx', index=False)

if __name__ == "__main__":
    main()

Rows: 1334
   year  month produced_material  produced_material_production_type  \
0  2024      1             10000                               8002   
1  2024      1             50000                               8002   
2  2024      1             50000                               8002   
3  2024      1             50000                               8002   
4  2024      1             80070                               8007   

  produced_material_release_type  produced_material_quantity  \
0                            FIN                         990   
1                           PROD                         859   
2                           PROD                         859   
3                           PROD                         859   
4                           PROD                         929   

  component_material  component_material_production_type  \
0              50000                              8002.0   
1              80070                              8007.0 

In [13]:
print(df_agg.groupby('produced_material')['component_material'].apply(list).to_dict())

{101: [802], 802: [803], 803: [804], 804: [805], 10000: [50000], 10001: [50001], 10002: [50002], 10003: [50003], 10004: [50004], 10005: [50005], 10006: [50006], 10007: [50007], 10008: [50008], 10009: [50009], 50000: [80070], 50001: [80071], 50002: [80072], 50003: [80073], 50004: [80074], 50005: [80075], 50006: [80076], 50007: [80077], 50008: [80078], 50009: [80079], 80010: [80000], 80011: [80001], 80012: [80002], 80013: [80003], 80014: [80004], 80015: [80005], 80016: [80006], 80017: [80007], 80018: [80008], 80019: [80009], 80070: [80010], 80071: [80011], 80072: [80012], 80073: [80013], 80074: [80014], 80075: [80015], 80076: [80016], 80077: [80017], 80078: [80018], 80079: [80019]}


In [27]:
print(fin_materials)
             

{101, 10000, 10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009}


In [25]:
print(df_agg[['produced_material', 'produced_material_release_type']]
            .drop_duplicates('produced_material')
            .set_index('produced_material')['produced_material_release_type'].to_dict())

{101: 'FIN', 802: 'PROD', 803: 'PROD', 804: 'PROD', 10000: 'FIN', 10001: 'FIN', 50000: 'PROD', 50001: 'PROD', 80010: 'PROD', 80011: 'PROD', 80070: 'PROD', 80071: 'PROD', 10002: 'FIN', 10003: 'FIN', 10004: 'FIN', 10005: 'FIN', 10006: 'FIN', 10009: 'FIN', 50002: 'PROD', 50003: 'PROD', 50004: 'PROD', 50005: 'PROD', 50006: 'PROD', 50009: 'PROD', 80012: 'PROD', 80013: 'PROD', 80014: 'PROD', 80015: 'PROD', 80016: 'PROD', 80019: 'PROD', 80072: 'PROD', 80073: 'PROD', 80074: 'PROD', 80075: 'PROD', 80076: 'PROD', 80079: 'PROD', 10007: 'FIN', 10008: 'FIN', 50007: 'PROD', 50008: 'PROD', 80017: 'PROD', 80018: 'PROD', 80077: 'PROD', 80078: 'PROD'}


In [29]:
print(df_agg[['produced_material', 'produced_material_production_type']]
            .drop_duplicates('produced_material')
            .set_index('produced_material')['produced_material_production_type'].to_dict())

{101: 81, 802: 82, 803: 83, 804: 84, 10000: 8002, 10001: 8002, 50000: 8002, 50001: 8002, 80010: 8001, 80011: 8001, 80070: 8007, 80071: 8007, 10002: 8002, 10003: 8002, 10004: 8002, 10005: 8002, 10006: 8002, 10009: 8002, 50002: 8002, 50003: 8002, 50004: 8002, 50005: 8002, 50006: 8002, 50009: 8002, 80012: 8001, 80013: 8001, 80014: 8001, 80015: 8001, 80016: 8001, 80019: 8001, 80072: 8007, 80073: 8007, 80074: 8007, 80075: 8007, 80076: 8007, 80079: 8007, 10007: 8002, 10008: 8002, 50007: 8002, 50008: 8002, 80017: 8001, 80018: 8001, 80077: 8007, 80078: 8007}


In [31]:
print(df_agg[['component_material', 'component_material_release_type']]
            .drop_duplicates('component_material')
            .set_index('component_material')['component_material_release_type'].to_dict())

{802: 'PROD', 803: 'PROD', 804: 'PROD', 805: 'PROD', 50000: 'PROD', 50001: 'PROD', 80070: 'PROD', 80071: 'PROD', 80000: 'PROD', 80001: 'PROD', 80010: 'PROD', 80011: 'PROD', 50002: 'PROD', 50003: 'PROD', 50004: 'PROD', 50005: 'PROD', 50006: 'PROD', 50009: 'PROD', 80072: 'PROD', 80073: 'PROD', 80074: 'PROD', 80075: 'PROD', 80076: 'PROD', 80079: 'PROD', 80002: 'PROD', 80003: 'PROD', 80004: 'PROD', 80005: 'PROD', 80006: 'PROD', 80009: 'PROD', 80012: 'PROD', 80013: 'PROD', 80014: 'PROD', 80015: 'PROD', 80016: 'PROD', 80019: 'PROD', 50007: 'PROD', 50008: 'PROD', 80077: 'PROD', 80078: 'PROD', 80007: 'PROD', 80008: 'PROD', 80017: 'PROD', 80018: 'PROD'}


In [33]:
print(df_agg[['component_material', 'component_material_production_type']]
            .drop_duplicates('component_material')
            .set_index('component_material')['component_material_production_type'].to_dict())

{802: 82.0, 803: 83.0, 804: 84.0, 805: 85.0, 50000: 8002.0, 50001: 8002.0, 80070: 8007.0, 80071: 8007.0, 80000: 8000.0, 80001: 8000.0, 80010: 8001.0, 80011: 8001.0, 50002: 8002.0, 50003: 8002.0, 50004: 8002.0, 50005: 8002.0, 50006: 8002.0, 50009: 8002.0, 80072: 8007.0, 80073: 8007.0, 80074: 8007.0, 80075: 8007.0, 80076: 8007.0, 80079: 8007.0, 80002: 8000.0, 80003: 8000.0, 80004: 8000.0, 80005: 8000.0, 80006: 8000.0, 80009: 8000.0, 80012: 8001.0, 80013: 8001.0, 80014: 8001.0, 80015: 8001.0, 80016: 8001.0, 80019: 8001.0, 50007: 8002.0, 50008: 8002.0, 80077: 8007.0, 80078: 8007.0, 80007: 8000.0, 80008: 8000.0, 80017: 8001.0, 80018: 8001.0}


In [35]:
print(df_agg.groupby(['produced_material', 'year', 'plant_id'])['produced_material_quantity'].sum().to_dict())

{(101, 2000, 'PLANT_15'): 1.0, (802, 2000, 'PLANT_15'): 11.0, (803, 2000, 'PLANT_15'): 111.0, (804, 2000, 'PLANT_15'): 1111.0, (10000, 2024, 'RLT_10'): 11708.0, (10001, 2024, 'RLT_10'): 12023.0, (10002, 2024, 'RLT_14'): 12067.0, (10003, 2024, 'RLT_14'): 12091.0, (10004, 2024, 'RLT_14'): 12091.0, (10005, 2024, 'RLT_14'): 12023.0, (10006, 2024, 'RLT_14'): 12067.0, (10007, 2024, 'RLT_16'): 12023.0, (10008, 2024, 'RLT_16'): 12067.0, (10009, 2024, 'RLT_14'): 12023.0, (50000, 2024, 'RLT_10'): 9538.0, (50001, 2024, 'RLT_10'): 9487.0, (50002, 2024, 'RLT_14'): 9603.0, (50003, 2024, 'RLT_14'): 9448.0, (50004, 2024, 'RLT_14'): 9448.0, (50005, 2024, 'RLT_14'): 9487.0, (50006, 2024, 'RLT_14'): 9603.0, (50007, 2024, 'RLT_16'): 9487.0, (50008, 2024, 'RLT_16'): 9603.0, (50009, 2024, 'RLT_14'): 9487.0, (80010, 2024, 'RLT_10'): 21013.0, (80011, 2024, 'RLT_10'): 21300.0, (80012, 2024, 'RLT_14'): 21410.0, (80013, 2024, 'RLT_14'): 21521.0, (80014, 2024, 'RLT_14'): 21521.0, (80015, 2024, 'RLT_14'): 21300.0,

In [39]:
print( df_agg.set_index(['produced_material', 'component_material', 'year', 'plant_id'])['component_material_quantity'].to_dict())

{(101, 802, 2000, 'PLANT_15'): 11.0, (802, 803, 2000, 'PLANT_15'): 111.0, (803, 804, 2000, 'PLANT_15'): 1111.0, (804, 805, 2000, 'PLANT_15'): 11111.0, (10000, 50000, 2024, 'RLT_10'): 11708.0, (10001, 50001, 2024, 'RLT_10'): 12023.0, (50000, 80070, 2024, 'RLT_10'): 11303.0, (50001, 80071, 2024, 'RLT_10'): 10759.0, (80010, 80000, 2024, 'RLT_10'): 23360.0, (80011, 80001, 2024, 'RLT_10'): 24730.0, (80070, 80010, 2024, 'RLT_10'): 41769.0, (80071, 80011, 2024, 'RLT_10'): 42650.0, (10002, 50002, 2024, 'RLT_14'): 12067.0, (10003, 50003, 2024, 'RLT_14'): 12091.0, (10004, 50004, 2024, 'RLT_14'): 12091.0, (10005, 50005, 2024, 'RLT_14'): 12023.0, (10006, 50006, 2024, 'RLT_14'): 12067.0, (10009, 50009, 2024, 'RLT_14'): 12023.0, (50002, 80072, 2024, 'RLT_14'): 11027.0, (50003, 80073, 2024, 'RLT_14'): 10806.0, (50004, 80074, 2024, 'RLT_14'): 10806.0, (50005, 80075, 2024, 'RLT_14'): 10759.0, (50006, 80076, 2024, 'RLT_14'): 11027.0, (50009, 80079, 2024, 'RLT_14'): 10759.0, (80012, 80002, 2024, 'RLT_14'

In [71]:
print(df_agg.groupby('produced_material')['component_material'].apply(list).to_dict().get(101)[:])

[802]


In [45]:
print(df_agg.groupby('produced_material')['component_material'].apply(list).to_dict().get(802)[:])

[803]


In [47]:
print(df_agg.groupby('produced_material')['component_material'].apply(list).to_dict().get(803)[:])

[804]


In [49]:
print(df_agg.groupby('produced_material')['component_material'].apply(list).to_dict().get(804)[:])

[805]


In [53]:
print(df_agg.groupby('produced_material')['component_material'].apply(list).to_dict().get(805,'Unknown')[:])

Unknown
